notebook for saving discharge objects in folders as cache

In [1]:
from pathlib import Path

import pandas as pd



from pha_lib.io import load_test_folder



# Set folders to process and overwrite behavior.

selected_folders = [

    "2025-04-29--10-19",

    "2025-04-29--10-24",

]

overwrite = False



base_data_dir = Path("data")

results = []



for folder_name in selected_folders:

    folder_path = base_data_dir / folder_name

    cache_dir = folder_path / "cache"

    cache_file = cache_dir / "discharge_object.pkl"



    if not folder_path.exists() or not folder_path.is_dir():

        status = "failed"

        error_msg = "missing or invalid folder"

        print(f"[{status.upper()}] {folder_name}: {error_msg}")

        results.append({

            "folder": folder_name,

            "status": status,

            "cache_file": str(cache_file),

            "size_bytes": None,

            "error": error_msg,

        })

        continue



    if cache_file.exists() and not overwrite:

        size_bytes = cache_file.stat().st_size

        status = "skipped"

        print(f"[{status.upper()}] {folder_name}: cache exists -> {cache_file}")

        results.append({

            "folder": folder_name,

            "status": status,

            "cache_file": str(cache_file),

            "size_bytes": size_bytes,

            "error": "",

        })

        continue



    try:

        discharge = load_test_folder(folder_path, discharge_id=folder_name)

        cache_dir.mkdir(parents=True, exist_ok=True)

        pd.to_pickle(discharge, cache_file)

        size_bytes = cache_file.stat().st_size if cache_file.exists() else None

        status = "loaded"

        print(f"[{status.upper()}] {folder_name}: cached -> {cache_file}")

        results.append({

            "folder": folder_name,

            "status": status,

            "cache_file": str(cache_file),

            "size_bytes": size_bytes,

            "error": "",

        })

    except Exception as exc:

        status = "failed"

        print(f"[{status.upper()}] {folder_name}: {exc}")

        results.append({

            "folder": folder_name,

            "status": status,

            "cache_file": str(cache_file),

            "size_bytes": None,

            "error": str(exc),

        })



loaded_count = sum(1 for r in results if r["status"] == "loaded")

skipped_count = sum(1 for r in results if r["status"] == "skipped")

failed_count = sum(1 for r in results if r["status"] == "failed")



print("\nSummary table:")

print("folder | source_status | cache_file_path | cache_size_bytes")

print("-" * 100)

for row in results:

    size_text = "-" if row["size_bytes"] is None else str(row["size_bytes"])

    print(f"{row['folder']} | {row['status']} | {row['cache_file']} | {size_text}")



print("\nFinal totals:")

print(f"loaded: {loaded_count}")

print(f"skipped: {skipped_count}")

print(f"failed: {failed_count}")


[LOADED] 2025-04-29--10-19: cached -> data\2025-04-29--10-19\cache\discharge_object.pkl
[LOADED] 2025-04-29--10-24: cached -> data\2025-04-29--10-24\cache\discharge_object.pkl

Summary table:
folder | source_status | cache_file_path | cache_size_bytes
----------------------------------------------------------------------------------------------------
2025-04-29--10-19 | loaded | data\2025-04-29--10-19\cache\discharge_object.pkl | 8047058
2025-04-29--10-24 | loaded | data\2025-04-29--10-24\cache\discharge_object.pkl | 8112610

Final totals:
loaded: 2
skipped: 0
failed: 0


In [2]:
import numpy as np

import pandas as pd



from pha_lib.io import load_test_folder





def compare_discharges(cached, fresh):

    """Return (is_equal, details) for two Discharge objects."""

    details = []

    ok = True



    # Compare top-level metadata first so failures are easy to read.

    if cached.discharge_id != fresh.discharge_id:

        ok = False

        details.append(f"discharge_id mismatch: {cached.discharge_id} != {fresh.discharge_id}")



    if not np.isclose(cached.frame_dt_s, fresh.frame_dt_s):

        ok = False

        details.append(f"frame_dt_s mismatch: {cached.frame_dt_s} != {fresh.frame_dt_s}")



    if set(cached.channels.keys()) != set(fresh.channels.keys()):

        ok = False

        details.append(

            f"channel keys mismatch: {sorted(cached.channels.keys())} != {sorted(fresh.channels.keys())}"

        )

        return ok, details



    # Compare each channel array-by-array to ensure object content is unchanged.

    for ch in sorted(cached.channels.keys()):

        cch = cached.channels[ch]

        fch = fresh.channels[ch]



        if cch.channel_id != fch.channel_id:

            ok = False

            details.append(f"channel {ch}: channel_id mismatch")



        if not np.array_equal(cch.frame_numbers, fch.frame_numbers):

            ok = False

            details.append(f"channel {ch}: frame_numbers mismatch")



        if not np.allclose(cch.energy_eV, fch.energy_eV, equal_nan=True):

            ok = False

            details.append(f"channel {ch}: energy_eV mismatch")



        if not np.allclose(cch.spectra, fch.spectra, equal_nan=True):

            ok = False

            details.append(f"channel {ch}: spectra mismatch")



    return ok, details





comparison_rows = []



for folder_name in selected_folders:

    folder_path = base_data_dir / folder_name

    cache_file = folder_path / "cache" / "discharge_object.pkl"



    if not folder_path.exists() or not folder_path.is_dir():

        print(f"[FAILED] {folder_name}: missing or invalid folder")

        comparison_rows.append(

            {

                "folder": folder_name,

                "compare_status": "failed",

                "cache_file": str(cache_file),

                "details": "missing or invalid folder",

            }

        )

        continue



    if not cache_file.exists():

        print(f"[FAILED] {folder_name}: cache not found -> {cache_file}")

        comparison_rows.append(

            {

                "folder": folder_name,

                "compare_status": "failed",

                "cache_file": str(cache_file),

                "details": "cache not found",

            }

        )

        continue



    try:

        cached_discharge = pd.read_pickle(cache_file)

        fresh_discharge = load_test_folder(folder_path, discharge_id=folder_name)

        is_equal, details = compare_discharges(cached_discharge, fresh_discharge)



        status = "match" if is_equal else "different"

        details_text = "OK" if is_equal else "; ".join(details)



        print(f"[{status.upper()}] {folder_name}: {details_text}")

        comparison_rows.append(

            {

                "folder": folder_name,

                "compare_status": status,

                "cache_file": str(cache_file),

                "details": details_text,

            }

        )

    except Exception as exc:

        print(f"[FAILED] {folder_name}: {exc}")

        comparison_rows.append(

            {

                "folder": folder_name,

                "compare_status": "failed",

                "cache_file": str(cache_file),

                "details": str(exc),

            }

        )





comparison_df = pd.DataFrame(comparison_rows)



match_count = int((comparison_df["compare_status"] == "match").sum()) if not comparison_df.empty else 0

different_count = int((comparison_df["compare_status"] == "different").sum()) if not comparison_df.empty else 0

failed_count = int((comparison_df["compare_status"] == "failed").sum()) if not comparison_df.empty else 0



print("\nComparison summary table:")

if comparison_df.empty:

    print("No rows")

else:

    print(comparison_df.to_string(index=False))



print("\nFinal totals:")

print(f"match: {match_count}")

print(f"different: {different_count}")

print(f"failed: {failed_count}")


[MATCH] 2025-04-29--10-19: OK
[MATCH] 2025-04-29--10-24: OK

Comparison summary table:
           folder compare_status                                        cache_file details
2025-04-29--10-19          match data\2025-04-29--10-19\cache\discharge_object.pkl      OK
2025-04-29--10-24          match data\2025-04-29--10-24\cache\discharge_object.pkl      OK

Final totals:
match: 2
different: 0
failed: 0


In [3]:
import pandas as pd



from pha_lib.io import load_test_folder



# Process every folder left in data that was not in the earlier selected_folders list.

all_data_folders = sorted([p.name for p in base_data_dir.iterdir() if p.is_dir()])

already_processed = set(selected_folders)

remaining_folders = [name for name in all_data_folders if name not in already_processed]



overwrite_remaining = False

cache_filename = "discharge_object.pkl"



print(f"Remaining folders: {len(remaining_folders)}")



if "compare_discharges" not in globals():

    raise RuntimeError("compare_discharges is not defined. Run Cell 2 first.")





# --- Step 1: export caches for remaining folders ---

export_rows = []



for folder_name in remaining_folders:

    folder_path = base_data_dir / folder_name

    cache_dir = folder_path / "cache"

    cache_file = cache_dir / cache_filename



    if not folder_path.exists() or not folder_path.is_dir():

        print(f"[FAILED-EXPORT] {folder_name}: missing or invalid folder")

        export_rows.append(

            {

                "folder": folder_name,

                "export_status": "failed",

                "cache_file": str(cache_file),

                "details": "missing or invalid folder",

            }

        )

        continue



    if cache_file.exists() and not overwrite_remaining:

        print(f"[SKIPPED-EXPORT] {folder_name}: cache exists -> {cache_file}")

        export_rows.append(

            {

                "folder": folder_name,

                "export_status": "skipped",

                "cache_file": str(cache_file),

                "details": "cache exists",

            }

        )

        continue



    try:

        discharge = load_test_folder(folder_path, discharge_id=folder_name)

        cache_dir.mkdir(parents=True, exist_ok=True)

        pd.to_pickle(discharge, cache_file)

        print(f"[LOADED-EXPORT] {folder_name}: cached -> {cache_file}")

        export_rows.append(

            {

                "folder": folder_name,

                "export_status": "loaded",

                "cache_file": str(cache_file),

                "details": "cached",

            }

        )

    except Exception as exc:

        print(f"[FAILED-EXPORT] {folder_name}: {exc}")

        export_rows.append(

            {

                "folder": folder_name,

                "export_status": "failed",

                "cache_file": str(cache_file),

                "details": str(exc),

            }

        )



export_df = pd.DataFrame(export_rows)





# --- Step 2: read-through and compare cache vs fresh load ---

compare_rows = []



for folder_name in remaining_folders:

    folder_path = base_data_dir / folder_name

    cache_file = folder_path / "cache" / cache_filename



    if not folder_path.exists() or not folder_path.is_dir():

        print(f"[FAILED-COMPARE] {folder_name}: missing or invalid folder")

        compare_rows.append(

            {

                "folder": folder_name,

                "compare_status": "failed",

                "cache_file": str(cache_file),

                "details": "missing or invalid folder",

            }

        )

        continue



    if not cache_file.exists():

        print(f"[FAILED-COMPARE] {folder_name}: cache not found -> {cache_file}")

        compare_rows.append(

            {

                "folder": folder_name,

                "compare_status": "failed",

                "cache_file": str(cache_file),

                "details": "cache not found",

            }

        )

        continue



    try:

        cached_discharge = pd.read_pickle(cache_file)

        fresh_discharge = load_test_folder(folder_path, discharge_id=folder_name)

        is_equal, details = compare_discharges(cached_discharge, fresh_discharge)



        status = "match" if is_equal else "different"

        details_text = "OK" if is_equal else "; ".join(details)



        print(f"[{status.upper()}-COMPARE] {folder_name}: {details_text}")

        compare_rows.append(

            {

                "folder": folder_name,

                "compare_status": status,

                "cache_file": str(cache_file),

                "details": details_text,

            }

        )

    except Exception as exc:

        print(f"[FAILED-COMPARE] {folder_name}: {exc}")

        compare_rows.append(

            {

                "folder": folder_name,

                "compare_status": "failed",

                "cache_file": str(cache_file),

                "details": str(exc),

            }

        )



compare_df = pd.DataFrame(compare_rows)





print("\nExport summary table:")

if export_df.empty:

    print("No rows")

else:

    print(export_df.to_string(index=False))



print("\nExport totals:")

print(f"loaded: {int((export_df['export_status'] == 'loaded').sum()) if not export_df.empty else 0}")

print(f"skipped: {int((export_df['export_status'] == 'skipped').sum()) if not export_df.empty else 0}")

print(f"failed: {int((export_df['export_status'] == 'failed').sum()) if not export_df.empty else 0}")



print("\nCompare summary table:")

if compare_df.empty:

    print("No rows")

else:

    print(compare_df.to_string(index=False))



print("\nCompare totals:")

print(f"match: {int((compare_df['compare_status'] == 'match').sum()) if not compare_df.empty else 0}")

print(f"different: {int((compare_df['compare_status'] == 'different').sum()) if not compare_df.empty else 0}")

print(f"failed: {int((compare_df['compare_status'] == 'failed').sum()) if not compare_df.empty else 0}")


Remaining folders: 16
[LOADED-EXPORT] 2025-04-29--10-42: cached -> data\2025-04-29--10-42\cache\discharge_object.pkl
[LOADED-EXPORT] 2025-04-29--10-49: cached -> data\2025-04-29--10-49\cache\discharge_object.pkl
[LOADED-EXPORT] 2025-04-29--11-17: cached -> data\2025-04-29--11-17\cache\discharge_object.pkl
[LOADED-EXPORT] 2025-04-29--11-19: cached -> data\2025-04-29--11-19\cache\discharge_object.pkl
[LOADED-EXPORT] 2025-04-29--11-23: cached -> data\2025-04-29--11-23\cache\discharge_object.pkl
[LOADED-EXPORT] 2025-04-29--11-32: cached -> data\2025-04-29--11-32\cache\discharge_object.pkl
[LOADED-EXPORT] 2025-04-29--11-37: cached -> data\2025-04-29--11-37\cache\discharge_object.pkl
[LOADED-EXPORT] 2025-04-29--12-14: cached -> data\2025-04-29--12-14\cache\discharge_object.pkl
[LOADED-EXPORT] 2025-04-29--12-28: cached -> data\2025-04-29--12-28\cache\discharge_object.pkl
[LOADED-EXPORT] 2025-04-29--12-50: cached -> data\2025-04-29--12-50\cache\discharge_object.pkl
[LOADED-EXPORT] 2025-04-29--

KeyboardInterrupt: 

In [3]:
from pathlib import Path
base_data_dir = Path("data")

In [4]:
#cell for changeing discharge object to 1st version
from datetime import datetime

import pandas as pd

from pha_lib.model import DISCHARGE_SCHEMA_VERSION

# Target values to stamp onto every cached Discharge object.
new_created_at = datetime(2026, 6, 6, 17, 0, 0)
new_schema_version = DISCHARGE_SCHEMA_VERSION  # current version (1)

cache_filename = "discharge_object.pkl"
update_rows = []

# Walk every data folder and update its cached Discharge, if one exists.
for folder_path in sorted(p for p in base_data_dir.iterdir() if p.is_dir()):
    folder_name = folder_path.name
    cache_file = folder_path / "cache" / cache_filename

    # Skip folders that have no cached object to update.
    if not cache_file.exists():
        continue

    try:
        # Load the cached object, overwrite the two fields, then re-save it.
        discharge = pd.read_pickle(cache_file)
        discharge.created_at = new_created_at
        discharge.schema_version = new_schema_version
        pd.to_pickle(discharge, cache_file)

        print(f"[UPDATED] {folder_name}: created_at={new_created_at}, schema_version={new_schema_version}")
        update_rows.append({
            "folder": folder_name,
            "status": "updated",
            "cache_file": str(cache_file),
            "created_at": str(new_created_at),
            "schema_version": new_schema_version,
            "error": "",
        })
    except Exception as exc:
        print(f"[FAILED] {folder_name}: {exc}")
        update_rows.append({
            "folder": folder_name,
            "status": "failed",
            "cache_file": str(cache_file),
            "created_at": "",
            "schema_version": "",
            "error": str(exc),
        })

update_df = pd.DataFrame(update_rows)

print("\nUpdate summary table:")
if update_df.empty:
    print("No cached objects found")
else:
    print(update_df.to_string(index=False))

print("\nFinal totals:")
print(f"updated: {int((update_df['status'] == 'updated').sum()) if not update_df.empty else 0}")
print(f"failed: {int((update_df['status'] == 'failed').sum()) if not update_df.empty else 0}")


[UPDATED] 2025-04-29--10-19: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--10-24: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--10-42: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--10-49: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--11-17: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--11-19: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--11-23: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--11-32: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--11-37: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--12-14: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--12-28: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--12-50: created_at=2026-06-06 17:00:00, schema_version=1
[UPDATED] 2025-04-29--12-58: created_at=2026-06-06 17:00:00, sch

In [5]:
from dataclasses import fields, is_dataclass

import pandas as pd

from pha_lib.model import Discharge, EnergyChannelData, DISCHARGE_SCHEMA_VERSION


def check_structure(obj, cls):
    """Compare an object's attributes against a dataclass's defined fields."""
    # Field names the current code defines for this class.
    expected = {f.name for f in fields(cls)}
    # Attributes the loaded (cached) object actually carries.
    actual = set(vars(obj).keys())

    missing = expected - actual   # fields the current class has but the cache lacks
    extra = actual - expected     # leftover fields from an older layout
    return missing, extra


cache_filename = "discharge_object.pkl"

for folder_path in sorted(p for p in base_data_dir.iterdir() if p.is_dir()):
    cache_file = folder_path / "cache" / cache_filename
    if not cache_file.exists():
        continue

    discharge = pd.read_pickle(cache_file)

    # 1) Top-level Discharge structure.
    missing, extra = check_structure(discharge, Discharge)

    # 2) Nested EnergyChannelData objects (check the first channel as a sample).
    ch_missing, ch_extra = set(), set()
    if discharge.channels:
        first_ch = next(iter(discharge.channels.values()))
        ch_missing, ch_extra = check_structure(first_ch, EnergyChannelData)

    same = not (missing or extra or ch_missing or ch_extra)
    print(f"{folder_path.name}: {'OK' if same else 'DIFFERENT'}")
    if not same:
        print(f"   Discharge   missing={missing} extra={extra}")
        print(f"   ChannelData missing={ch_missing} extra={ch_extra}")

2025-04-29--10-19: OK
2025-04-29--10-24: OK
2025-04-29--10-42: OK
2025-04-29--10-49: OK
2025-04-29--11-17: OK
2025-04-29--11-19: OK
2025-04-29--11-23: OK
2025-04-29--11-32: OK
2025-04-29--11-37: OK
2025-04-29--12-14: OK
2025-04-29--12-28: OK
2025-04-29--12-50: OK
2025-04-29--12-58: OK
2025-04-29--13-17: OK
2025-04-29--13-51: OK
test: OK
